# C-R60k — Return-v3, S4 dài 1.200 slot

1. Tạo **Dataset mới** bằng ZIP source Return-v3, Add Input, chọn CPU rồi Run All.
2. Giữ map BS5, launch B, ba vị trí trạm, công suất sạc, reward, QMIX và curriculum.
3. Returning tìm đường quanh ô đang bị chiếm, ngưỡng về sạc tính theo đường hiện tại.
4. Không đủ năng lượng: UAV đứng lại, không radio và vẫn chiếm ô. Game tiếp tục; không tự cứu.
5. Trạm cho phép tất cả UAV cùng sạc, mỗi UAV vẫn nhận 5 pin/slot; không có queue sạc.

**Đây là train fresh một C-R/seed551, 60k train +72k validation. Không nối checkpoint v2.**
S1 0–3k, S2 3–6k, S3 6–12k: horizon150; S4 12–60k:40 game×1200.
Validation0/12/24/36/48/60k, mỗi mốc10 game×1200. Không tự mở rộng120k.

Preflight có bộ đếm bền vững, cap512 diagnostic-train +4096 control/eval; main seed không chạy trong preflight.
Run có raw từ slot đầu, resume state ở game boundary, export một lần cuối phiên.
`MAX_SESSION_SECONDS` mặc định7 giờ, dừng ở game boundary; đây không phải cam kết thời lượng Kaggle.

Đường có thể bị chặn hoàn toàn hoặc thay đổi ở slot sau. Bản sửa không bảo đảm mọi UAV luôn về được trạm.
Khi chưa có đường, ngưỡng dùng khoảng cách hình học làm lower bound và ghi rõ no-route; không kích Return ngay chỉ vì UAV đầy pin tạm bị vây ở launch.


In [ ]:
from pathlib import Path
SOURCE_PATH = None  # None: tìm tự động; hoặc đường dẫn ZIP/thư mục source trong /kaggle/input
RESUME_INPUT_DIR = None  # thư mục chứa export ZIPs + EXPORT_INDEX.json, hoặc merged result root
OUTPUT_ROOT = Path("/kaggle/working/bs5_cr_h1200_return_v3_seed551")
DEVICE = "cpu"  # chỉ đổi thành "cuda" TRƯỚC prepare run mới
RUN_PREFLIGHT = True
RUN_MAIN = True
MAX_SESSION_SECONDS = 7 * 3600  # None: không đặt giới hạn phiên bằng notebook


In [ ]:
import os, sys, zipfile, hashlib, json, shutil, subprocess, time
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
MARKER = "configs/bs5_cr_h1200_60k_return_v3.json"

def safe_extract(z, target):
    target=Path(target).resolve()
    for member in z.infolist():
        dest=(target/member.filename).resolve()
        if not dest.is_relative_to(target): raise ValueError("Unsafe ZIP member: "+member.filename)
        if (member.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError("ZIP symlink rejected")
    z.extractall(target)

if SOURCE_PATH is None:
    directories=[p.parent.parent for p in INPUT.rglob("bs5_cr_h1200_60k_return_v3.json")
                 if p.parent.name=="configs" and (p.parent.parent/"scripts/training/run_bs5_cr_h1200_return.py").exists()]
    archives=[]
    for f in INPUT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(f) as z:
                if "uav_thesis/"+MARKER in z.namelist(): archives.append(f)
        except zipfile.BadZipFile: pass
    candidates=directories+archives
    if len(candidates)!=1:
        raise RuntimeError("Chọn SOURCE_PATH chính xác từ candidates: "+str(candidates))
    SOURCE_PATH=candidates[0]
SOURCE_PATH=Path(SOURCE_PATH)
SOURCE_ROOT=WORK/"bs5_return_v3_source"/"uav_thesis"
if SOURCE_PATH.is_dir():
    shutil.copytree(SOURCE_PATH,SOURCE_ROOT,dirs_exist_ok=True)
else:
    with zipfile.ZipFile(SOURCE_PATH) as z:
        if z.testzip() is not None: raise RuntimeError("Source ZIP CRC failed")
        safe_extract(z,SOURCE_ROOT.parent)
assert (SOURCE_ROOT/MARKER).exists()
sys.path.insert(0,str(SOURCE_ROOT))
os.chdir(SOURCE_ROOT)
for namespace in ("src", "scripts"):
    loaded=sys.modules.get(namespace)
    if loaded is not None:
        origins=list(getattr(loaded,"__path__",[]))
        if getattr(loaded,"__file__",None): origins.append(loaded.__file__)
        if any(not Path(origin).resolve().is_relative_to(SOURCE_ROOT.resolve()) for origin in origins):
            raise RuntimeError("Kernel đang giữ source cũ. Restart Session rồi Run All notebook Return-v3.")
import numpy as np, torch
try:
    import pytest
except ImportError as e:
    raise RuntimeError("Runtime thiếu pytest. Cài pytest rồi khởi động lại; notebook không tự đổi Torch/CUDA.") from e
print({"source":str(SOURCE_ROOT),"python":sys.version,"numpy":np.__version__,"torch":torch.__version__,"device":DEVICE})


In [ ]:
from scripts.training import run_bs5_cr_h1200_return as runner
runner.setup_runtime(DEVICE,1)
identity=runner.source_identity()
PREFLIGHT=WORK/"bs5_return_v3_preflight"
receipt=PREFLIGHT/"receipt.json"
fingerprint={"source":identity["sha256"],"runtime":runner.runtime(DEVICE)}
if receipt.exists():
    previous=json.loads(receipt.read_text())
    if previous.get("fingerprint")!=fingerprint or previous.get("status")!="passed":
        raise RuntimeError("Preflight khác source/runtime hoặc chưa PASS. Giữ log; không tự chạy lại compute.")
    print("Dùng preflight PASS cùng source/runtime:",previous["charged"])
elif RUN_PREFLIGHT:
    PREFLIGHT.mkdir(parents=True,exist_ok=True)
    result=subprocess.run([sys.executable,"-B","scripts/verification/preflight_bs5_return.py",
        "--output",str(PREFLIGHT),"--device",DEVICE],cwd=SOURCE_ROOT,text=True,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    (PREFLIGHT/"pytest.log").write_text(result.stdout)
    print(result.stdout)
    if result.returncode:raise RuntimeError("Preflight thất bại; giữ log, main chưa chạy.")
    previous=json.loads(receipt.read_text())
    assert previous["status"]=="passed" and previous["fingerprint"]==fingerprint
else:
    raise RuntimeError("Cần preflight PASS trước main; bật RUN_PREFLIGHT=True.")


In [ ]:
# Optional restore. Do not copy only weights/best.pt and call that a resume.
if RESUME_INPUT_DIR is not None:
    src=Path(RESUME_INPUT_DIR)
    if OUTPUT_ROOT.exists(): raise RuntimeError("Output đã tồn tại; không chép đè khi restore.")
    OUTPUT_ROOT.mkdir(parents=True)
    if (src/"protocol.json").exists() and (src/"resume.pt").exists():
        shutil.copytree(src,OUTPUT_ROOT,dirs_exist_ok=True)
    else:
        indices=list(src.rglob("EXPORT_INDEX.json"))
        if len(indices)!=1: raise RuntimeError("Cần đúng một EXPORT_INDEX.json; chọn RESUME_INPUT_DIR cụ thể.")
        index=json.loads(indices[0].read_text())
        for item in index["files"]:
            found=list(src.rglob(item["path"]))
            if len(found)!=1: raise RuntimeError("Thiếu/không rõ companion: "+item["path"])
            f=found[0]
            if hashlib.sha256(f.read_bytes()).hexdigest()!=item["sha256"]: raise RuntimeError("Companion hash failed")
            if item["kind"]=="independent_zip":
                with zipfile.ZipFile(f) as z: safe_extract(z,OUTPUT_ROOT)
            else:
                dest=(OUTPUT_ROOT/item["restore_to"]).resolve()
                if not dest.is_relative_to(OUTPUT_ROOT.resolve()): raise ValueError("unsafe restore_to")
                dest.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(f,dest)
    assert (OUTPUT_ROOT/"resume.pt").exists(), "Thiếu full training resume state"

if RUN_MAIN:
    if not OUTPUT_ROOT.exists(): runner.prepare(OUTPUT_ROOT,DEVICE)
    p=runner.read_json(OUTPUT_ROOT/"protocol.json")
    assert p.get("return_version")=="bs5_energy_return_v3", "Không resume env cũ bằng runner Return-v3"
    assert not p["diagnostic"] and p["total_steps"]==60000 and p["training_seed"]==551
    assert p["device"]==DEVICE and p["source_sha256"]==identity["sha256"]
    # Copy the receipt only when creating this main lineage, not on a restored hashed export.
    if not (OUTPUT_ROOT/"preflight_receipt.json").exists():
        shutil.copy2(receipt,OUTPUT_ROOT/"preflight_receipt.json")
    runner.run(OUTPUT_ROOT,resume=(OUTPUT_ROOT/"resume.pt").exists(),max_seconds=MAX_SESSION_SECONDS)
    print(runner.read_json(OUTPUT_ROOT/"status.json"))
else:
    print("RUN_MAIN=False: main chưa chạy.")


In [ ]:
# Export once after complete or graceful boundary pause. No policy re-evaluation.
if RUN_MAIN and (OUTPUT_ROOT/"commit.json").exists():
    EXPORT=runner.export(OUTPUT_ROOT)
    print("Export:",EXPORT)
    from IPython.display import display, FileLink
    for f in sorted(EXPORT.iterdir()):
        if f.is_file(): display(FileLink(str(f)))
    print("Gửi lại review_small.zip, toàn bộ audit_light ZIPs và EXPORT_INDEX.json để phân tích offline.")
    print("Giữ thêm mọi repro_full file + raw companions để resume; không chỉ giữ best weights.")


## Đọc kết quả và chạy tiếp

- `review_small.zip`: bảng, curve, protocol và báo cáo nhanh.
- `audit_light*.zip`: đầy đủ raw để audit; giải nén cùng một thư mục. Analyzer không cần Torch/checkpoint.
- `repro_full*` cùng audit files, `EXPORT_INDEX.json`: restore full state và history. Nếu có file `.pt` riêng, đặt vào `restore_to` theo index.
- `paused_at_game_boundary` là dừng an toàn. Tải các output, nén toàn bộ thư mục export thành một ZIP ngoài rồi upload Dataset (giữ nguyên các ZIP companion bên trong), khai báo `RESUME_INPUT_DIR` ở phiên sau, dùng cùng source/runtime/device.
- Lỗi/kill giữa game có thể có compute chưa commit. Runner giữ ledger và chặn tự rollback/rerun vượt cap; gửi status/ledger để kiểm tra, không sửa/xóa chúng để chạy tiếp.
- Dừng tự động tại60k. Nối120k cần quyết định sau khi xem bằng chứng.

Đây là **một seed**, validation đã dùng trước đây. Không suy ra hội tụ hoặc ưu thế curriculum/direct/C-R/C-N mới. Best được chọn bằng mean utility toàn1.200 slot; last full state phục vụ resume.

Return-v3: xem `return_wait_uav_slots`, `return_no_route_uav_slots`, `return_detour_uav_slots`, `return_stranded_uav_slots`, `return_prevented_energy_moves`. Analyzer độc lập kiểm đường, năng lượng, lifecycle và collision từ raw; không cần checkpoint hay NumPy/Torch. Giữ các run cũ làm tham chiếu; không suy ra lợi ích chính sách từ fixture điều khiển cố định.
